# Phase 5 — Final Exam: base vs fine-tuned on dev.json

dev.jsonl (1,034 examples) meets the model for the first and only time.
Both models generate greedily; both prediction files go through the verified
harness (exact match + execution accuracy). Budget: ~1.5–2.5 h total.
Runs fine interactively or as a background commit.

In [ ]:
# CELL 1 — config
import os
os.environ["REPO_URL"]        = "https://github.com/Rick-Roy-JC/text-to-sql-qlora-v2.git"
os.environ["DATA_DATASET"]    = "/kaggle/input/datasets/aritraroy3/spider-prepared-v2"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# CELL 2 — hardware + pinned installs
!nvidia-smi --query-gpu=name,compute_cap --format=csv
%pip install -q unsloth==2026.6.9 trl==0.24.0 peft==0.19.1 transformers==5.5.0

In [ ]:
# CELL 3 — repo, data, AND the database mount check (before any GPU spend)
import os
%cd /kaggle/working
if not os.path.exists("text-to-sql-qlora-v2"):
    !git clone $REPO_URL
%cd text-to-sql-qlora-v2
!git pull
!mkdir -p data && cp $DATA_DATASET/*.jsonl data/

# Find where the database folder actually mounted (nesting demon check):
!find $DATA_DATASET -name "*.sqlite" | head -3
# Set DB_ROOT to the parent of the db_id folders, based on what printed above:
os.environ["DB_ROOT"] = os.path.join(os.environ["DATA_DATASET"], "database")
!ls $DB_ROOT | head -5

In [ ]:
# CELL 4 — adapter: pull from the repo copy you committed
import os
assert os.path.exists("adapters/final_r16/adapter_config.json"), \
    "adapters/final_r16 not found in repo — commit it and push, then git pull"
print("adapter found in repo ✓")

In [ ]:
# CELL 5 — smoke the generator on 8 examples (~2 min) before the full runs
!python src/generate.py --data data/dev.jsonl --out /tmp/preds_smoke.txt \
    --adapter adapters/final_r16 --limit 8
!cat /tmp/preds_smoke.txt
# Eyeball: 8 lines, each looking like SQL. Garbage here -> STOP, report to Claude.

In [ ]:
# CELL 6 — full generation, BASE model (~30-60 min)
!python src/generate.py --data data/dev.jsonl --out preds_base.txt

In [ ]:
# CELL 7 — full generation, FINE-TUNED (~30-60 min)
!python src/generate.py --data data/dev.jsonl --out preds_ft.txt \
    --adapter adapters/final_r16

In [ ]:
# CELL 8 — THE NUMBERS. Harness both prediction files (CPU, fast).
print("========== BASE Phi-3-mini ==========")
!python eval/harness.py --data data/dev.jsonl --predictions preds_base.txt \
    --db_root $DB_ROOT --out report_base.jsonl
print()
print("========== FINE-TUNED r=16 ==========")
!python eval/harness.py --data data/dev.jsonl --predictions preds_ft.txt \
    --db_root $DB_ROOT --out report_ft.jsonl

In [ ]:
# CELL 9 — package everything for the repo
!zip -q /kaggle/working/phase5_results.zip preds_base.txt preds_ft.txt \
    report_base.jsonl report_ft.jsonl
!ls -la /kaggle/working/phase5_results.zip
print("Download from Output tab -> commit to repo under eval/results/")